In [2]:
# suppress tensorflow logging, usually not useful unless you are having problems with tensorflow or accessing gpu
# it seems necessary to have this environment variable set before tensorflow is imported, or else it doesn't take effect
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 

# imports generally useful throughout the notebook
# usually all imports should happen at the top of a notebook, but in
# these notebooks where the purpose is to show how to use the Keras API
# the relevant imports will happen in the cells where the API is discussed
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import datetime
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# global settings for notebook output and images
plt.rcParams['figure.figsize'] = (8, 8) # set default figure size, 10in by 8in
np.set_printoptions(precision=4, suppress=True)

In [3]:
# import project defined modules / functions used in this notebook
# ensure that the src directory where project modules are found is on
# the PYTHONPATH
import sys
sys.path.append("../src")

# assignment function imports for doctests and github autograding
# these are required for assignment autograding
from nndl import vectorize_samples, plot_history

In [4]:
# if want to restrict to cpu or gpu, configure visible device for rest of notebook to use
dev = tf.config.list_physical_devices()
print('Physical Devices : ', dev)

tf.config.set_visible_devices(dev[0])
#tf.config.set_visible_devices(dev[1])
#dev = tf.config.list_logical_devices()
print('Available Devices : ', dev)

Physical Devices :  [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Available Devices :  [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


# Chapter 10: Deep Learning for Timeseries

Supporting materials for:

Chollet (2021). *Deep Learning with Python*. 2nd ed. Manning Publications Co.
[Amazon](https://www.amazon.com/Learning-Python-Second-Fran%C3%A7ois-Chollet/dp/1617296864/ref=sr_1_1?crid=32NFM2SBCJVQQ)

In this section we turn to deep learning architecutures for modeling timeseries data.  A timeseries can be any data obtained
via measurements at regular intervals, like the daily price of a stock, hourly electricity consumption, or processing speech input.
A common task for a timeseries is forecasting, predicting what will hapen next in the series.  Recurrent layers are designed
specifically to process and predict time series data.

## 10.3 Understanding Recurrent Neural Networks

- All of the neural networks you've seen so far have no memory.
  - Each input shown is processed independently with no state kept between inputs.
  - For a dense network, we flatten five days of data into a single large vector: **feedforward networks**.
- Biological systems use memory
  - while reading you process each word one-by-one while keeping memories of what came before.
- A **recurrent neural network** (RNN) adopts the same principle.
  - Processes sequences by iterating through the sequence elements and maintaining a **state**

To make the pseudocode absolutely unambiguous, let's write a naive NumPy
implementation of the forward pass of the simple RNN:

In [6]:
# numpy implementation of a simple RNN forward pass
# number of timesteps in the input sequence (in our weather example this was 5 * 24 = 120
timesteps = 100

# dimensionality of the input feature space and the output feature space that is generated
input_features = 32
output_features = 64

# some random inputs, just to demonstrate
inputs = np.random.random((timesteps, input_features))

# initial state an all-zero vector
state_t = np.zeros((output_features,))

# random weight matrices, would be initialized to small nonzero for W and U and 0 for bias
W = np.random.random((output_features, input_features))
U = np.random.random((output_features, output_features))
b = np.random.random((output_features,))

# final output is sequence of the sequence of outputs calculated
successive_outputs = []

# input_t is a vector of shape (input_features,)
for input_t in inputs:
    # combines the input with the current state (the previous output)
    # to obtain the current output, tanh is the activation function here
    output_t = np.tanh(np.dot(W, input_t) + np.dot(U, state_t) + b)
    # store list of outputs during this sequence
    successive_outputs.append(output_t)
    # updates the state of the network for the next timestep
    state_t = output_t

# final output is a rank-2 tensor of shape (timesteps, output_features)
final_output_sequence = np.stack(successive_outputs, axis=0)

print(inputs.shape)
print(final_output_sequence.shape)

(100, 32)
(100, 64)


In summary, an RNN is a `for` loop that reuses quantities computed during the previous iteration of the loop.

### 10.3.1 A recurrent layer in Keras

The naive pseudocode we implemented in NumPy corresponds to an actual Keras layer, the `SimpleRNN` layer.

- All recurrent layers in Keras `SimpleRNN`, `LSTM` and `GRU` can be run in two different modes
  1. they can return the full sequences of successive outputs for each time step like we did before,
     a rank-3 tensor of shape `(batch_size, timesteps, output_features)`
  2. or they can return only the last output for each input sequence a rank-2 tensor of shape
     `(batch_size, output_features)`

First an RNN that returns only its last output step:

In [7]:
num_features = 14
steps = 120
inputs = keras.Input(shape=(steps, num_features))
# note that return_sequences = False is the default
outputs = layers.SimpleRNN(16, return_sequences=False)(inputs)
print(outputs.shape)

(None, 16)


While the following returns the full state of sequences while iterating over the input sequence:

In [8]:
num_features = 14
steps = 120
inputs = keras.Input(shape=(steps, num_features))
# note that return_sequences = False is the default, have to specify to get full history of sequences
outputs = layers.SimpleRNN(16, return_sequences=True)(inputs)
print(outputs.shape)

(None, 120, 16)


It's sometimes useful to stack several recurrent layers one after the other in order to increase the
representation power of a network.

In that case you have to get all of the intermediate layers to return a full sequence of outputs.


In [10]:
inputs = keras.Input(shape=(steps, num_features))
rnn1 = layers.SimpleRNN(16, return_sequences=True)(inputs)
rnn2 = layers.SimpleRNN(32, return_sequences=True)(rnn1)
outputs = layers.SimpleRNN(16)(rnn2)
print(rnn1.shape)
print(rnn2.shape)
print(outputs.shape)

(None, 120, 16)
(None, 120, 32)
(None, 16)


## Summary

<font color='blue'>
    
- A **recurrent neural network** (RNN) processes sequences by iterating through the sequence elements and maingaining a **state**
- The state of the RNN is reset between processing two different sequences.  What changes is that the data point is no longer
  processed in a single step, the network internall **loops** over sequence elements.
- All reurrent layers in Keras can be run in two different modes: return the full sequence history of successive outputs, or only return the last output.
- In practice you'll rarely work with the `SimpleRNN` it suffers from equivalent of vanishing gradients.
- LSTM for example adds carry flow, which act similar to residual connections.
- Keep in mind what the LSTM cell is meant to do: allow past information to be reinjected at a later time, thus fighting the vanishing-gradient problem.